<a href="https://colab.research.google.com/github/lohithharish798-creator/capstone-project-part-4/blob/main/capstone_part_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import userdata
import requests

In [31]:
OPENROUTER_API_KEY =userdata.get('lohithkey')
print(f"key preview:{OPENROUTER_API_KEY[:8]}{'*'* 24}")
#

key preview:sk-or-v1************************


In [12]:
import joblib
import requests
import json
from jsonschema import validate
import numpy as np # Import numpy for creating dummy data

# 1. Load your best model
model = joblib.load("/content/sample_data/random_forest_model (1).pkl")

# 2. Generate 64-feature dummy vectors for testing (replace with your actual dataset features)
# Creating 3 samples, each with 64 random features.
X_samples = np.random.rand(3, 64).tolist()

# 3. Define JSON schema for validation
schema = {
    "type": "object",
    "properties": {
        "predicted_label": {"type": "string"},
        "predicted_probability": {"type": "number"},
        "top_feature": {"type": "string"},
        "feature_importance": {"type": "number"},
        "next_best_class": {"type": "string"}
    },
    "required": ["predicted_label", "predicted_probability",
                 "top_feature", "feature_importance", "next_best_class"]
}

# 4. Loop through samples
# API_KEY = "lohithkey" # This line is no longer needed

for x in X_samples:
    pred_class = model.predict([x])[0]
    pred_prob = max(model.predict_proba([x])[0])

    # Build prompt for LLM
    prompt = f"""
    The model predicted class {pred_class} with probability {pred_prob}.
    Input features: {x}
    Explain this prediction in JSON with fields:
    predicted_label, predicted_probability, top_feature, feature_importance, next_best_class.
    """

    # 5. Call OpenRouter API (instead of OpenAI API)
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions", # Changed endpoint to OpenRouter
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "HTTP-Referer": "YOUR_APP_NAME", # Replace with your app name/domain for OpenRouter
            "X-Title": "YOUR_APP_TITLE" # Replace with your app title for OpenRouter
        },
        json={
            "model": "gpt-4", # You can choose other models supported by OpenRouter
            "messages": [{"role": "user", "content": prompt}]
        }
    )

    # Debugging: Print status code and full response content
    print(f"API Response Status Code: {response.status_code}")
    response_json = response.json()
    print(f"API Response JSON: {json.dumps(response_json, indent=2)}")

    # 6. Parse and validate JSON
    # Check if 'choices' key exists before accessing it
    if 'choices' in response_json and len(response_json['choices']) > 0:
        explanation = response_json["choices"][0]["message"]["content"]
        try:
            parsed = json.loads(explanation)
            validate(instance=parsed, schema=schema)
            print("Validated Explanation:", parsed)
        except json.JSONDecodeError as e:
            print(f"JSON decoding failed for explanation: {e}")
            print(f"Raw explanation content: {explanation}")
        except Exception as e:
            print(f"Validation failed: {e}")
    else:
        print("Error: 'choices' key not found in API response or choices list is empty.")
        print(f"Full API response for reference: {json.dumps(response_json, indent=2)}")


API Response Status Code: 402
API Response JSON: {
  "error": {
    "message": "This request requires more credits, or fewer max_tokens. You requested up to 4096 tokens, but can only afford 666. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account",
    "code": 402,
    "metadata": {
      "provider_name": null,
      "previous_errors": [
        {
          "code": 402,
          "message": "This request requires more credits, or fewer max_tokens. You requested up to 4096 tokens, but can only afford 666. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account"
        }
      ]
    }
  },
  "user_id": "user_3FUzGob1n5eYo9LBPoNHNZeV8KA"
}
Error: 'choices' key not found in API response or choices list is empty.
Full API response for reference: {
  "error": {
    "message": "This request requires more credits, or fewer max_tokens. You requested up to 4096 tokens, but can only afford 666. To increase, visit https://openrou

In [32]:
OPENROUTER_API_KEY =userdata.get('lohithkey')
print(f"key preview:{OPENROUTER_API_KEY[:8]}{'*'* 24}")
#

key preview:sk-or-v1************************


In [36]:
import os, requests, json
from jsonschema import validate

# Removed: OPENROUTER_API_KEY["userdata.get"] = "lohithkey" as OPENROUTER_API_KEY is already defined globally

schema_A = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "email": {"type": "string"},
        "phone": {"type": "string"},
        "address": {"type": "string"}
    },
    "required": ["name", "email", "phone", "address"]
}

def call_llm(system_prompt, user_prompt, api_key, temperature=0.0, max_tokens=512):
    # Correctly use the passed api_key
    # Change URL to OpenRouter endpoint
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "YOUR_APP_NAME", # Replace with your app name/domain for OpenRouter
        "X-Title": "YOUR_APP_TITLE" # Replace with your app title for OpenRouter
    }
    payload = {
        "model": "gpt-4",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    response = requests.post(url, headers=headers, json=payload)

    # Debugging: Print status code and full response content
    print(f"API Response Status Code: {response.status_code}")
    response_json = response.json()
    print(f"API Response JSON: {json.dumps(response_json, indent=2)}")

    # Check if 'choices' key exists before accessing it
    if 'choices' in response_json and len(response_json['choices']) > 0:
        return response_json["choices"][0]["message"]["content"]
    else:
        # Handle the case where 'choices' key is missing or empty
        print("Error: 'choices' key not found in API response or choices list is empty.")
        print(f"Full API response for reference: {json.dumps(response_json, indent=2)}")
        return ""

record = {"domain":"example.com","name":"Alice","email":"alice@example.com","phone":"1234567890","address":"Chennai"}
prompt = f"Input record:\n{record}\nExtract into JSON schema."
# Pass the globally defined OPENROUTER_API_KEY
output = call_llm("You are a structured data extractor. Output only valid JSON.", prompt, OPENROUTER_API_KEY)

if output:
    try:
        parsed = json.loads(output)
        validate(instance=parsed, schema=schema_A)
        print("Validated JSON:", parsed)
    except json.JSONDecodeError as e:
        print(f"JSON decoding failed for explanation: {e}")
        print(f"Raw output content: {output}")
    except Exception as e:
        print(f"Validation failed: {e}")
else:
    print("LLM did not return a valid response.")

API Response Status Code: 200
API Response JSON: {
  "id": "gen-1783961102-hi5qp20mVflYa9XqPeSc",
  "object": "chat.completion",
  "created": 1783961102,
  "model": "openai/gpt-4",
  "provider": "OpenAI",
  "system_fingerprint": null,
  "service_tier": null,
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "stop",
      "native_finish_reason": "stop",
      "message": {
        "role": "assistant",
        "content": "{\n  \"domain\": \"example.com\",\n  \"name\": \"Alice\",\n  \"email\": \"alice@example.com\",\n  \"phone\": \"1234567890\",\n  \"address\": \"Chennai\"\n}",
        "refusal": null,
        "reasoning": null
      }
    }
  ],
  "usage": {
    "prompt_tokens": 68,
    "completion_tokens": 44,
    "total_tokens": 112,
    "cost": 0.00468,
    "is_byok": false,
    "prompt_tokens_details": {
      "cached_tokens": 0,
      "cache_write_tokens": 0,
      "audio_tokens": 0,
      "video_tokens": 0
    },
    "cost_details": {
      "upstr

In [40]:
schema_B = {
    "type": "object",
    "properties": {
        "risk_tier": {"type": "string"},
        "flag_for_review": {"type": "boolean"},
        "primary_signal": {"type": "string"},
        "confidences": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": ["risk_tier","flag_for_review","primary_signal","confidences","recommended_action"]
}

records = [
    {"domain":"abc.com","revenue":5000},
    {"domain":"xyz.com","revenue":200},
    {"domain":"pqr.com","revenue":10000}
]

for rec in records:
    # Define the rubric criteria within the user prompt
    rubric_criteria = f"""
    Based on the input record, assess the following:
    - risk_tier: 'high_risk' if revenue < 1000, 'medium_risk' if revenue between 1000 and 5000, 'low_risk' if revenue > 5000.
    - flag_for_review: True if revenue is less than 500, otherwise False.
    - primary_signal: Explain the main reason for the risk tier (e.g., 'Very high revenue', 'Extremely low revenue', 'Moderate revenue').
    - confidences: Your confidence level in the assessment, choose from 'high', 'medium', 'low'.
    - recommended_action: A brief action recommendation based on the assessment (e.g., 'Monitor closely', 'Further investigation required', 'No action needed').
    """

    user_prompt = f"Input record:\n{rec}\n{rubric_criteria}\nOutput the assessment in JSON format matching schema_B. Only output the JSON."

    # Pass the OPENROUTER_API_KEY to the call_llm function
    output = call_llm("You are a record assessor. Score each record against rubric criteria and output valid JSON.", user_prompt, OPENROUTER_API_KEY)

    if output:
        # Clean the output by removing markdown code block fences if they exist
        if output.strip().startswith('```json') and output.strip().endswith('```'):
            output = output.strip()[len('```json'):-len('```')].strip()

        try:
            parsed = json.loads(output)
            validate(instance=parsed, schema=schema_B)
            print("Validated Assessment:", parsed)
        except json.JSONDecodeError as e:
            print(f"JSON decoding failed for explanation: {e}")
            print(f"Raw output content: {output}")
        except Exception as e:
            print(f"Validation failed: {e}")
    else:
        print("LLM did not return a valid response.")

API Response Status Code: 200
API Response JSON: {
  "id": "gen-1783961435-AtgghzysXVFtVEiVvTff",
  "object": "chat.completion",
  "created": 1783961435,
  "model": "openai/gpt-4",
  "provider": "OpenAI",
  "system_fingerprint": null,
  "service_tier": null,
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "stop",
      "native_finish_reason": "stop",
      "message": {
        "role": "assistant",
        "content": "{\n    \"domain\": \"abc.com\",\n    \"risk_tier\": \"medium_risk\",\n    \"flag_for_review\": false,\n    \"primary_signal\": \"Moderate revenue\",\n    \"confidences\": \"high\",\n    \"recommended_action\": \"Monitor closely\"\n}",
        "refusal": null,
        "reasoning": null
      }
    }
  ],
  "usage": {
    "prompt_tokens": 226,
    "completion_tokens": 56,
    "total_tokens": 282,
    "cost": 0.01014,
    "is_byok": false,
    "prompt_tokens_details": {
      "cached_tokens": 0,
      "cache_write_tokens": 0,
      "audio

In [41]:
schema_A = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "email": {"type": "string"},
        "phone": {"type": "string"},
        "address": {"type": "string"},
        "age": {"type": "number"}
    },
    "required": ["name", "email", "phone", "address", "age"]
}

In [43]:
import json
from jsonschema import validate, ValidationError

records = [
    {"domain":"abc.com","name":"Alice","email":"alice@example.com","phone":"1234567890","address":"Chennai","age":25},
    {"domain":"xyz.com","name":"Bob","email":"bob@example.com","phone":"9876543210","address":"Delhi","age":30},
    {"domain":"pqr.com","name":"Charlie","email":"charlie@example.com","phone":"5555555555","address":"Mumbai","age":40}
]

for rec in records:
    user_prompt = f"Input record:\n{rec}\nExtract into JSON schema."
    # Pass the OPENROUTER_API_KEY as the third argument to call_llm
    response = call_llm("You are a structured data extractor. Output only valid JSON.", user_prompt, OPENROUTER_API_KEY)
    try:
        parsed = json.loads(response.strip())
        validate(instance=parsed, schema=schema_A)
        print("Valid JSON:", parsed)
    except json.JSONDecodeError as e:
        print("JSON decode error:", e)
    except ValidationError as e:
        print("Schema validation error:", e)
        print("Fallback:", {k: None for k in schema_A["required"]})

API Response Status Code: 402
API Response JSON: {
  "error": {
    "message": "This request requires more credits, or fewer max_tokens. You requested up to 512 tokens, but can only afford 435. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account",
    "code": 402,
    "metadata": {
      "provider_name": null,
      "previous_errors": [
        {
          "code": 402,
          "message": "This request requires more credits, or fewer max_tokens. You requested up to 512 tokens, but can only afford 435. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account"
        }
      ]
    }
  },
  "user_id": "user_3FUzGob1n5eYo9LBPoNHNZeV8KA"
}
Error: 'choices' key not found in API response or choices list is empty.
Full API response for reference: {
  "error": {
    "message": "This request requires more credits, or fewer max_tokens. You requested up to 512 tokens, but can only afford 435. To increase, visit https://openrouter

In [44]:
import re

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.[a-zA-Z0-9._-]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

In [45]:
user_prompt = f"Input record:\n{record}\nExtract into JSON schema."

if has_pii(user_prompt):
    print("Input blocked. PII detected.")
else:
    response = call_llm(system_prompt, user_prompt)
    # continue with JSON parsing + validation

Input blocked. PII detected.


In [47]:
test_input1 = "My email is alice@example.com"
if has_pii(test_input1):
    print("Blocked:", test_input1)

# Case 2: No PII (allowed)
test_input2 = "This record has revenue=5000 and domain=abc.com"
if has_pii(test_input2):
    print("Blocked:", test_input2)
else:
    response = call_llm("You are a record assessor.", test_input2, OPENROUTER_API_KEY)
    print("LLM Output:", response)

Blocked: My email is alice@example.com
API Response Status Code: 402
API Response JSON: {
  "error": {
    "message": "This request requires more credits, or fewer max_tokens. You requested up to 512 tokens, but can only afford 435. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account",
    "code": 402,
    "metadata": {
      "provider_name": null,
      "previous_errors": [
        {
          "code": 402,
          "message": "This request requires more credits, or fewer max_tokens. You requested up to 512 tokens, but can only afford 435. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account"
        }
      ]
    }
  },
  "user_id": "user_3FUzGob1n5eYo9LBPoNHNZeV8KA"
}
Error: 'choices' key not found in API response or choices list is empty.
Full API response for reference: {
  "error": {
    "message": "This request requires more credits, or fewer max_tokens. You requested up to 512 tokens, but can only afford 435

In [48]:
import json
from jsonschema import validate, ValidationError

# Schema with 5 scalar fields
schema_A = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "email": {"type": "string"},
        "phone": {"type": "string"},
        "address": {"type": "string"},
        "age": {"type": "number"}
    },
    "required": ["name", "email", "phone", "address", "age"]
}

# Guardrail function
import re
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.[a-zA-Z0-9._-]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

# Test inputs
records = [
    {"domain":"abc.com","name":"Alice","email":"alice@example.com","phone":"1234567890","address":"Chennai","age":25},
    {"domain":"xyz.com","name":"Bob","email":"bob@example.com","phone":"9876543210","address":"Delhi","age":30},
    {"domain":"pqr.com","name":"Charlie","email":"charlie@example.com","phone":"5555555555","address":"Mumbai","age":40}
]

for rec in records:
    user_prompt = f"Input record:\n{rec}\nExtract into JSON schema."
    print("\nInput:", rec)

    # Guardrail check
    if has_pii(user_prompt):
        print("Guardrail: Blocked (PII detected)")
        continue

    # Call LLM
    response = call_llm("You are a structured data extractor. Output only valid JSON.", user_prompt)
    print("Raw LLM Output:", response)

    # Validation
    try:
        parsed = json.loads(response.strip())
        validate(instance=parsed, schema=schema_A)
        print("Validation: PASS ✅", parsed)
    except json.JSONDecodeError as e:
        print("Validation: FAIL ❌ (JSON decode error)", e)
    except ValidationError as e:
        print("Validation: FAIL ❌ (Schema error)", e)


Input: {'domain': 'abc.com', 'name': 'Alice', 'email': 'alice@example.com', 'phone': '1234567890', 'address': 'Chennai', 'age': 25}
Guardrail: Blocked (PII detected)

Input: {'domain': 'xyz.com', 'name': 'Bob', 'email': 'bob@example.com', 'phone': '9876543210', 'address': 'Delhi', 'age': 30}
Guardrail: Blocked (PII detected)

Input: {'domain': 'pqr.com', 'name': 'Charlie', 'email': 'charlie@example.com', 'phone': '5555555555', 'address': 'Mumbai', 'age': 40}
Guardrail: Blocked (PII detected)
